# Meteo prognoza — LightGBM 

Ova bilježnica koristi isti memorijski štedljiv pipeline kao prethodna verzija, ali omogućuje treniranje  modela `LightGBM`


## 1. Importi i konfiguracija

In [1]:
!pip install lightgbm
!pip install optuna optuna-integration


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Ako nedostaju paketi, pokreni: %pip install lightgbm xgboost
import gc
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import optuna
from optuna.pruners import MedianPruner
from optuna_integration.lightgbm import LightGBMPruningCallback

MODEL_NAME = 'LightGBM'

RANDOM_STATE = 42
HORIZONT = 12
VALIDATION_FRACTION = 0.10
BUFFER_STUPNJEVI = 0.3
CSV_PATH = 'ERA5_SPOJENO_2015_2023.csv'
OUTPUT_RESULTS = f'{MODEL_NAME.lower()}_rezultati_po_targetu.csv'

LAT_PULA, LON_PULA = 44.8666, 13.8496
LAT_RIJEKA, LON_RIJEKA = 45.3271, 14.4422

TARGETS = ['t2m_c', 'wind_speed_ms', 'swh', 'mwp', 'mwd', 'sst_c', 'msl_hpa', 'tcc', 'cape', 'blh']
LAG_VARS = ['t2m_c', 'msl_hpa', 'wind_speed_ms', 'tcc', 'cape']

# Ako nedostaje RAM-a, postavi npr. 300_000.
MAX_TRAIN_ROWS = None

## 2. Učitavanje podataka

In [5]:
df = pd.read_csv(CSV_PATH)
df['valid_time'] = pd.to_datetime(df['valid_time'])
df = df.sort_values(['latitude', 'longitude', 'valid_time']).reset_index(drop=True)

print(f'Ukupno redaka: {len(df):,}')
print(f'Broj stupaca: {df.shape[1]}')
print(f'Broj lokacija: {df[["latitude", "longitude"]].drop_duplicates().shape[0]}')
print(f'Period: {df["valid_time"].min()} do {df["valid_time"].max()}')
print('Duplikati:', df.duplicated(['valid_time', 'latitude', 'longitude']).sum())

Ukupno redaka: 3,367,224
Broj stupaca: 37
Broj lokacija: 42
Period: 2015-01-01 00:00:00 do 2023-12-31 23:00:00
Duplikati: 53928


## 3. Izdvajanje lokacija

In [6]:
locations = df[['latitude', 'longitude']].drop_duplicates().copy()
locations['dist_pula'] = np.sqrt((locations['latitude'] - LAT_PULA)**2 + (locations['longitude'] - LON_PULA)**2)
locations['dist_rijeka'] = np.sqrt((locations['latitude'] - LAT_RIJEKA)**2 + (locations['longitude'] - LON_RIJEKA)**2)
pula_grid = locations.loc[locations['dist_pula'].idxmin(), ['latitude', 'longitude']]
rijeka_grid = locations.loc[locations['dist_rijeka'].idxmin(), ['latitude', 'longitude']]
lat_pula_grid, lon_pula_grid = pula_grid['latitude'], pula_grid['longitude']
lat_rijeka_grid, lon_rijeka_grid = rijeka_grid['latitude'], rijeka_grid['longitude']

df['dist_pula'] = np.sqrt((df['latitude'] - LAT_PULA)**2 + (df['longitude'] - LON_PULA)**2)
df['dist_rijeka'] = np.sqrt((df['latitude'] - LAT_RIJEKA)**2 + (df['longitude'] - LON_RIJEKA)**2)
je_pula = (df['latitude'] == lat_pula_grid) & (df['longitude'] == lon_pula_grid)
je_rijeka = (df['latitude'] == lat_rijeka_grid) & (df['longitude'] == lon_rijeka_grid)
blizu = (df['dist_pula'] < BUFFER_STUPNJEVI) | (df['dist_rijeka'] < BUFFER_STUPNJEVI)

df_pula = df.loc[je_pula].copy()
df_rijeka = df.loc[je_rijeka].copy()
df_train = df.loc[~je_pula & ~je_rijeka & ~blizu].copy()

if MAX_TRAIN_ROWS is not None and len(df_train) > MAX_TRAIN_ROWS:
    df_train = df_train.sort_values(['valid_time', 'latitude', 'longitude']).iloc[:MAX_TRAIN_ROWS].copy()

print(f'Pula: {len(df_pula):,}')
print(f'Rijeka: {len(df_rijeka):,}')
print(f'Trening: {len(df_train):,}')
del locations
gc.collect()

Pula: 80,172
Rijeka: 80,172
Trening: 2,725,848


2132

## 4. Feature engineering

In [7]:
def add_features(data):
    data = data.sort_values(['latitude', 'longitude', 'valid_time']).copy()
    groups = data.groupby(['latitude', 'longitude'])

    for var in LAG_VARS:
        if var not in data.columns:
            continue
        data[f'{var}_lag3h'] = groups[var].shift(3)
        data[f'{var}_lag6h'] = groups[var].shift(6)
        data[f'{var}_trend3h'] = data[var] - data[f'{var}_lag3h']
        data[f'{var}_rolling6h_mean'] = groups[var].transform(lambda s: s.rolling(6, min_periods=1).mean())

    if 'wind_dir_deg' in data.columns:
        data['wind_dir_sin'] = np.sin(np.radians(data['wind_dir_deg']))
        data['wind_dir_cos'] = np.cos(np.radians(data['wind_dir_deg']))

    hour = data['valid_time'].dt.hour
    day = data['valid_time'].dt.dayofyear
    data['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    data['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    data['day_sin'] = np.sin(2 * np.pi * day / 365.25)
    data['day_cos'] = np.cos(2 * np.pi * day / 365.25)

    if {'t2m_c', 'd2m_c'}.issubset(data.columns):
        data['temp_dewpoint_diff'] = data['t2m_c'] - data['d2m_c']
    if 'msl_hpa' in data.columns:
        data['pressure_gradient'] = groups['msl_hpa'].diff()
    return data

df_train = add_features(df_train)
df_pula = add_features(df_pula)
df_rijeka = add_features(df_rijeka)
print('Broj stupaca nakon feature engineeringa:', df_train.shape[1])

Broj stupaca nakon feature engineeringa: 67


## 5. Future targeti

In [8]:
def add_future_targets(data):
    data = data.sort_values(['latitude', 'longitude', 'valid_time']).copy()
    groups = data.groupby(['latitude', 'longitude'])
    for target in TARGETS:
        if target in data.columns:
            data[f'{target}_future'] = groups[target].shift(-HORIZONT)
    return data

df_train = add_future_targets(df_train)
df_pula = add_future_targets(df_pula)
df_rijeka = add_future_targets(df_rijeka)
for target in TARGETS:
    col = f'{target}_future'
    if col in df_train.columns:
        print(f'{target}: {df_train[col].notna().sum():,} valjanih targeta')

t2m_c: 2,681,784 valjanih targeta
wind_speed_ms: 2,681,784 valjanih targeta
swh: 631,008 valjanih targeta
mwp: 631,008 valjanih targeta
mwd: 631,008 valjanih targeta
sst_c: 1,340,892 valjanih targeta
msl_hpa: 2,681,784 valjanih targeta
tcc: 2,681,784 valjanih targeta
cape: 2,681,784 valjanih targeta
blh: 2,681,784 valjanih targeta


## 6. Priprema featurea i model

In [ ]:
DROP_COLUMNS = ['valid_time', 'latitude', 'longitude', 'dist_pula', 'dist_rijeka', 'hour', 'day']

def prepare_X(data, feature_names=None, medians=None):
    drop_cols = [c for c in DROP_COLUMNS if c in data.columns]
    drop_cols += [c for c in data.columns if c.endswith('_future')]
    numeric_cols = data.select_dtypes(include=[np.number]).columns
    selected_cols = [c for c in numeric_cols if c not in drop_cols]

    if feature_names is not None:
        selected_cols = [c for c in feature_names if c in data.columns]

    X = data.loc[:, selected_cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    if medians is None:
        medians = X.median()
    X = X.fillna(medians)

    if feature_names is not None:
        X = X.reindex(columns=feature_names, fill_value=0)

    X = X.astype(np.float32)
    medians = medians.astype(np.float32)
    return X, medians

def make_model(name):
    if name == 'LightGBM':
        return lgb.LGBMRegressor(
            n_estimators=500, learning_rate=0.05, max_depth=10,
            num_leaves=31, min_child_samples=20,
            reg_alpha=0.5, reg_lambda=1.0,
            random_state=RANDOM_STATE, n_jobs=2, verbosity=-1
        )

  

    raise ValueError('MODEL_NAME mora biti LightGBM ')

## 7. Trening jednog targeta

In [10]:
def train_one_target(model_name, target, train_data, test_dict):
    target_col = f'{target}_future'
    if target_col not in train_data.columns:
        print(f'{target}: target ne postoji')
        return None

    valid_idx = train_data[target_col].notna()
    n_valid = int(valid_idx.sum())
    if n_valid < 1000:
        print(f'{target}: preskačem, samo {n_valid} valjanih redaka')
        return None

    target_data = train_data.loc[valid_idx]
    y = target_data[target_col].astype(np.float32)
    X, medians = prepare_X(target_data)
    feature_names = X.columns.tolist()

    cutoff = int(len(X) * (1 - VALIDATION_FRACTION))
    X_fit, X_val = X.iloc[:cutoff], X.iloc[cutoff:]
    y_fit, y_val = y.iloc[:cutoff], y.iloc[cutoff:]

    print(f'{MODEL_NAME}/{target}: treniranje na {len(X_fit):,} redaka i {len(feature_names)} featurea')
    model = make_model(model_name)
    model.fit(X_fit, y_fit)
    pred_val = model.predict(X_val)

    result = {
        'model': model_name,
        'target': target,
        'horizon_hours': HORIZONT,
        'MAE_val': mean_absolute_error(y_val, pred_val),
        'RMSE_val': np.sqrt(mean_squared_error(y_val, pred_val)),
        'R2_val': r2_score(y_val, pred_val),
        'n_train': len(X_fit),
        'n_validation': len(X_val),
        'n_features': len(feature_names)
    }

    for location, test_data in test_dict.items():
        test_valid_idx = test_data[target_col].notna()
        if test_valid_idx.sum() < 10:
            result[f'MAE_{location}'] = np.nan
            result[f'RMSE_{location}'] = np.nan
            result[f'R2_{location}'] = np.nan
            continue

        test_subset = test_data.loc[test_valid_idx]
        X_test, _ = prepare_X(test_subset, feature_names, medians)
        y_test = test_subset[target_col].astype(np.float32)
        pred_test = model.predict(X_test)
        result[f'MAE_{location}'] = mean_absolute_error(y_test, pred_test)
        result[f'RMSE_{location}'] = np.sqrt(mean_squared_error(y_test, pred_test))
        result[f'R2_{location}'] = r2_score(y_test, pred_test)

    print(f"{target}: MAE={result['MAE_val']:.3f}, RMSE={result['RMSE_val']:.3f}, R2={result['R2_val']:.3f}")
    del target_data, X, X_fit, X_val, y, y_fit, y_val, pred_val, model
    if 'X_test' in locals():
        del X_test
    if 'y_test' in locals():
        del y_test
    if 'pred_test' in locals():
        del pred_test
    gc.collect()
    return result

## 8. Pokretanje odabranog modela

In [ ]:
if MODEL_NAME not in ['LightGBM']:
    raise ValueError('MODEL_NAME mora biti LightGBM ')

test_dict = {'pula': df_pula, 'rijeka': df_rijeka}
results = []

for target in TARGETS:
    result = train_one_target(MODEL_NAME, target, df_train, test_dict)
    if result is not None:
        results.append(result)

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_RESULTS, index=False)
print('\n=== REZULTATI ===')
display(results_df)
print(f'Spremljeno u: {OUTPUT_RESULTS}')

LightGBM/t2m_c: treniranje na 2,413,605 redaka i 62 featurea
t2m_c: MAE=1.405, RMSE=1.847, R2=0.951
LightGBM/wind_speed_ms: treniranje na 2,413,605 redaka i 62 featurea
wind_speed_ms: MAE=0.781, RMSE=0.996, R2=0.369
LightGBM/swh: treniranje na 567,907 redaka i 62 featurea
swh: MAE=0.118, RMSE=0.176, R2=0.681
LightGBM/mwp: treniranje na 567,907 redaka i 62 featurea
mwp: MAE=0.270, RMSE=0.355, R2=0.809
LightGBM/mwd: treniranje na 567,907 redaka i 62 featurea
mwd: MAE=44.415, RMSE=66.726, R2=0.329
LightGBM/sst_c: treniranje na 1,206,802 redaka i 62 featurea
sst_c: MAE=0.148, RMSE=0.264, R2=0.998
LightGBM/msl_hpa: treniranje na 2,413,605 redaka i 62 featurea
msl_hpa: MAE=1.585, RMSE=2.121, R2=0.927
LightGBM/tcc: treniranje na 2,413,605 redaka i 62 featurea
tcc: MAE=0.221, RMSE=0.273, R2=0.481
LightGBM/cape: treniranje na 2,413,605 redaka i 62 featurea
cape: MAE=96.973, RMSE=239.954, R2=0.597
LightGBM/blh: treniranje na 2,413,605 redaka i 62 featurea
blh: MAE=180.918, RMSE=255.667, R2=0.717

,model,target,horizon_hours,MAE_val,RMSE_val,R2_val,n_train,n_validation,n_features,MAE_pula,RMSE_pula,R2_pula,MAE_rijeka,RMSE_rijeka,R2_rijeka
0,LightGBM,t2m_c,12,1.404994,1.847128,0.951372,2413605,268179,62,0.671788,0.903237,0.979575,1.040475,1.371684,0.967888
1,LightGBM,wind_speed_ms,12,0.780925,0.995601,0.369184,2413605,268179,62,1.481767,1.937002,0.568562,0.843112,1.108732,0.557465
2,LightGBM,swh,12,0.118167,0.176226,0.681185,567907,63101,62,NaN,NaN,NaN,NaN,NaN,NaN
3,LightGBM,mwp,12,0.270052,0.354822,0.809310,567907,63101,62,NaN,NaN,NaN,NaN,NaN,NaN
4,LightGBM,mwd,12,44.415030,66.726267,0.328806,567907,63101,62,NaN,NaN,NaN,NaN,NaN,NaN
5,LightGBM,sst_c,12,0.148391,0.264187,0.998048,1206802,134090,62,0.091082,0.150091,0.999211,NaN,NaN,NaN
6,LightGBM,msl_hpa,12,1.585445,2.121256,0.927215,2413605,268179,62,1.281886,1.719182,0.949690,1.316545,1.755338,0.946111
7,LightGBM,tcc,12,0.221071,0.272692,0.481340,2413605,268179,62,0.229531,0.278715,0.480468,0.203898,0.253253,0.554137
8,LightGBM,cape,12,96.972651,239.954453,0.597314,2413605,268179,62,147.086739,312.642504,0.804312,92.019671,204.062621,0.647494
9,LightGBM,blh,12,180.917539,255.667144,0.716832,2413605,268179,62,141.328375,197.221083,0.659514,159.183420,224.493577,0.719154


Spremljeno u: lightgbm_rezultati_po_targetu.csv


## 9. optuna za jedan target


In [15]:
# ============================================================
# OPTUNA LIGHTGBM ZA SVE TARGETE
# ============================================================

OPTUNA_TRIALS = 20
OPTUNA_MAX_ROWS = None

def objective_lightgbm(trial, target):
    target_col = f'{target}_future'

    valid_idx = df_train[target_col].notna()
    data = df_train.loc[valid_idx]

    if OPTUNA_MAX_ROWS is not None:
        data = (
            data
            .sort_values(['valid_time', 'latitude', 'longitude'])
            .iloc[:OPTUNA_MAX_ROWS]
        )

    y = data[target_col].astype(np.float32)
    X, medians = prepare_X(data)

    cutoff = int(len(X) * 0.90)

    X_fit = X.iloc[:cutoff]
    X_val = X.iloc[cutoff:]
    y_fit = y.iloc[:cutoff]
    y_val = y.iloc[cutoff:]

    params = {
        'objective': 'regression',
        'metric': 'l1',
        'verbosity': -1,
        'random_state': RANDOM_STATE,
        'n_jobs': 2,

        'n_estimators': trial.suggest_int(
            'n_estimators', 200, 800
        ),

        'learning_rate': trial.suggest_float(
            'learning_rate', 0.01, 0.15, log=True
        ),

        'num_leaves': trial.suggest_int(
            'num_leaves', 15, 127
        ),

        'max_depth': trial.suggest_int(
            'max_depth', 4, 14
        ),

        'min_child_samples': trial.suggest_int(
            'min_child_samples', 10, 100
        ),

        'subsample': trial.suggest_float(
            'subsample', 0.7, 1.0
        ),

        'colsample_bytree': trial.suggest_float(
            'colsample_bytree', 0.6, 1.0
        ),

        'reg_alpha': trial.suggest_float(
            'reg_alpha', 1e-8, 10.0, log=True
        ),

        'reg_lambda': trial.suggest_float(
            'reg_lambda', 1e-8, 10.0, log=True
        )
    }

    model = lgb.LGBMRegressor(**params)

    model.fit(
        X_fit,
        y_fit,
        eval_set=[(X_val, y_val)],
        eval_names=['valid_0'],
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=50,
                verbose=False
            )
        ]
    )

    pred = model.predict(X_val)

    return mean_absolute_error(y_val, pred)

## pokretanje OPtune  

In [16]:
# ============================================================
# POKRETANJE OPTUNE ZA SVE TARGETE
# ============================================================

optuna_rezultati = []
optuna_studies = {}

for target in TARGETS:
    print(f'\n===== OPTUNA: {target} =====')

    target_col = f'{target}_future'

    if target_col not in df_train.columns:
        print(f'{target}: target ne postoji, preskačem.')
        continue

    broj_validnih = df_train[target_col].notna().sum()

    if broj_validnih < 1000:
        print(
            f'{target}: premalo validnih podataka '
            f'({broj_validnih}), preskačem.'
        )
        continue

    study = optuna.create_study(
        direction='minimize',
        study_name=f'lightgbm_{target}',
        sampler=optuna.samplers.TPESampler(
            seed=RANDOM_STATE
        ),
        pruner=optuna.pruners.MedianPruner(
            n_startup_trials=5,
            n_warmup_steps=50
        )
    )

    study.optimize(
        lambda trial, current_target=target:
            objective_lightgbm(
                trial,
                current_target
            ),
        n_trials=OPTUNA_TRIALS,
        show_progress_bar=True
    )

    optuna_studies[target] = study

    najbolji_red = {
        'model': 'LightGBM',
        'target': target,
        'best_MAE': study.best_value,
        'n_trials': len(study.trials),
        **study.best_params
    }

    optuna_rezultati.append(najbolji_red)

    print(f'Najbolji MAE za {target}: {study.best_value:.4f}')
    print('Najbolji parametri:')
    print(study.best_params)

optuna_df = pd.DataFrame(optuna_rezultati)

optuna_df.to_csv(
    'optuna_lightgbm_svi_targeti.csv',
    index=False
)

print('\n=== OPTUNA REZULTATI ZA SVE TARGETE ===')
display(optuna_df)

print(
    '\nSpremljeno u: '
    'optuna_lightgbm_svi_targeti.csv'
)

[I 2026-08-19 21:05:32,181] A new study created in memory with name: lightgbm_t2m_c



===== OPTUNA: t2m_c =====


  0%|          | 0/20 [00:00<?, ?it/s]

c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[I 2026-08-19 21:07:10,199] Trial 0 finished with value: 1.1399912519195448 and parameters: {'n_estimators': 425, 'learning_rate': 0.13125830316209655, 'num_leaves': 97, 'max_depth': 10, 'min_child_samples': 24, 'subsample': 0.7467983561008608, 'colsample_bytree': 0.6232334448672797, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.002570603566117598}. Best is trial 0 with value: 1.1399912519195448.


...

[I 2026-08-20 02:19:47,099] Trial 18 finished with value: 186.173034364777 and parameters: {'n_estimators': 702, 'learning_rate': 0.09022688728529243, 'num_leaves': 108, 'max_depth': 4, 'min_child_samples': 46, 'subsample': 0.8968131503264609, 'colsample_bytree': 0.8158521383556735, 'reg_alpha': 5.0560307719444744e-05, 'reg_lambda': 1.866525267041869e-08}. Best is trial 12 with value: 138.86719809872017.


c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[I 2026-08-20 02:23:25,222] Trial 19 finished with value: 158.2037234068018 and parameters: {'n_estimators': 738, 'learning_rate': 0.03765110464989317, 'num_leaves': 114, 'max_depth': 12, 'min_child_samples': 97, 'subsample': 0.8591790707058076, 'colsample_bytree': 0.9113240518308489, 'reg_alpha': 8.435923682268883e-08, 'reg_lambda': 7.976989870150589e-06}. Best is trial 12 with value: 138.86719809872017.
Najbolji MAE za blh: 138.8672
Najbolji parametri:
{'n_estimators': 757, 'learning_rate': 0.1373669452343056, 'num_leaves': 123, 'max_depth': 11, 'min_child_samples': 52, 'subsample': 0.9615462369688978, 'colsample_bytree': 0.6828922252774745, 'reg_alpha': 0.0020781973056227185, 'reg_lambda': 0.008036141722925789}

=== OPTUNA REZULTATI ZA SVE TARGETE ===


,model,target,best_MAE,n_trials,n_estimators,learning_rate,num_leaves,max_depth,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,LightGBM,t2m_c,0.993751,20,757,0.137367,123,11,52,0.961546,0.682892,0.002078,0.008036
1,LightGBM,wind_speed_ms,0.642735,20,757,0.137367,123,11,52,0.961546,0.682892,0.002078,0.008036
2,LightGBM,swh,0.098296,20,757,0.137367,123,11,52,0.961546,0.682892,0.002078,0.008036
3,LightGBM,mwp,0.233492,20,757,0.137367,123,11,52,0.961546,0.682892,0.002078,0.008036
4,LightGBM,mwd,41.990450,20,697,0.084168,106,12,96,0.938063,0.684562,0.010837,0.000023
5,LightGBM,sst_c,0.140592,20,798,0.057551,46,14,58,0.884564,0.986765,0.001486,0.017475
6,LightGBM,msl_hpa,1.018551,20,757,0.137367,123,11,52,0.961546,0.682892,0.002078,0.008036
7,LightGBM,tcc,0.163016,20,757,0.137367,123,11,52,0.961546,0.682892,0.002078,0.008036
8,LightGBM,cape,74.234269,20,757,0.137367,123,11,52,0.961546,0.682892,0.002078,0.008036
9,LightGBM,blh,138.867198,20,757,0.137367,123,11,52,0.961546,0.682892,0.002078,0.008036



Spremljeno u: optuna_lightgbm_svi_targeti.csv
